In [131]:
import logging
import pathlib
import datetime
import polars as pl
from polars import Schema
from polars.datatypes import Boolean, Float64, Int64, List, String, Struct

ROOT_DIR_PATH = pathlib.Path(".").resolve().parent
DATA_DIR_PATH = ROOT_DIR_PATH / "data"
FAKE_GAMING_DATA_DIR = DATA_DIR_PATH / "fake_gaming_data"

log = logging.getLogger(__name__)


In [32]:
# Define the schema for the fake gaming data

schema = Schema(
    {
        "team_id": String,
        "name": String,
        "created_date": String,
        "ranking": Int64,
        "total_winnings": Int64,
        "members": List(
            Struct(
                {
                    "player_id": String,
                    "username": String,
                    "account_details": Struct(
                        {
                            "email": String,
                            "registration_date": String,
                            "premium_status": Boolean,
                            "country": String,
                            "language": String,
                        }
                    ),
                    "stats": Struct(
                        {
                            "level": Int64,
                            "experience": Int64,
                            "total_matches": Int64,
                            "win_rate": Float64,
                            "playtime_hours": Int64,
                            "achievements_completed": Int64,
                        }
                    ),
                    "inventory": Struct(
                        {
                            "currency": Struct({"premium": Int64, "standard": Int64}),
                            "items": List(
                                Struct(
                                    {
                                        "item_id": String,
                                        "name": String,
                                        "type": String,
                                        "rarity": String,
                                        "level_requirement": Int64,
                                        "stats": Struct(
                                            {
                                                "attack": Int64,
                                                "defense": Int64,
                                                "magic": Int64,
                                                "speed": Int64,
                                            }
                                        ),
                                    }
                                )
                            ),
                        }
                    ),
                    "achievements": List(
                        Struct(
                            {
                                "id": String,
                                "name": String,
                                "difficulty": String,
                                "completion_rate": Float64,
                                "points": Int64,
                                "date": String,
                            }
                        )
                    ),
                    "recent_matches": List(
                        Struct(
                            {
                                "match_id": String,
                                "game_mode": String,
                                "map": String,
                                "duration_minutes": Int64,
                                "date": String,
                                "stats": Struct(
                                    {
                                        "kills": Int64,
                                        "deaths": Int64,
                                        "assists": Int64,
                                        "damage_dealt": Int64,
                                        "healing_done": Int64,
                                        "accuracy": Float64,
                                        "headshot_percentage": Float64,
                                        "objectives_completed": Int64,
                                    }
                                ),
                                "rewards": Struct(
                                    {
                                        "experience": Int64,
                                        "currency": Int64,
                                        "items_dropped": List(
                                            Struct(
                                                {
                                                    "item_id": String,
                                                    "name": String,
                                                    "rarity": String,
                                                    "value": Int64,
                                                }
                                            )
                                        ),
                                    }
                                ),
                            }
                        )
                    ),
                }
            )
        ),
        "tournament_history": List(
            Struct(
                {
                    "tournament_id": String,
                    "name": String,
                    "placement": Int64,
                    "prize_money": Int64,
                    "matches_played": Int64,
                }
            )
        ),
    }
)


In [33]:
# Load the fake gaming data into a LazyFrame
data: pl.LazyFrame = pl.scan_ndjson(FAKE_GAMING_DATA_DIR / "data.json", schema=schema)

In [36]:
data.schema

/var/folders/n9/0hvfs68d1rg9vj676hndpzvh0000gn/T/ipykernel_21402/2326241270.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  data.schema


Schema([('team_id', String),
        ('name', String),
        ('created_date', String),
        ('ranking', Int64),
        ('total_winnings', Int64),
        ('members',
         List(Struct({'player_id': String, 'username': String, 'account_details': Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}), 'stats': Struct({'level': Int64, 'experience': Int64, 'total_matches': Int64, 'win_rate': Float64, 'playtime_hours': Int64, 'achievements_completed': Int64}), 'inventory': Struct({'currency': Struct({'premium': Int64, 'standard': Int64}), 'items': List(Struct({'item_id': String, 'name': String, 'type': String, 'rarity': String, 'level_requirement': Int64, 'stats': Struct({'attack': Int64, 'defense': Int64, 'magic': Int64, 'speed': Int64})}))}), 'achievements': List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})), 'recent_matches': List(Struct(

In [40]:
# Print the first 5 rows of the data
sample: pl.DataFrame = data.limit(5).collect()
sample.tail(1)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ team_id      ┆ name         ┆ created_date ┆ ranking ┆ total_winnin ┆ members      ┆ tournament_ │
│ ---          ┆ ---          ┆ ---          ┆ ---     ┆ gs           ┆ ---          ┆ history     │
│ str          ┆ str          ┆ str          ┆ i64     ┆ ---          ┆ list[struct[ ┆ ---         │
│              ┆              ┆              ┆         ┆ i64          ┆ 7]]          ┆ list[struct │
│              ┆              ┆              ┆         ┆              ┆              ┆ [5]]        │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ 7cf94e81-d7e ┆ Guild Scott- ┆ 2024-10-08   ┆ 253     ┆ 192516       ┆ [{"0741e111- ┆ [{"3e583592 │
│ e-4ecb-8545- ┆ Coleman      ┆              ┆         ┆              ┆ 0e5c-4524-be ┆ -4b94-4e29- │
│ 4cf559…      ┆              ┆              ┆         ┆              ┆ 09-6dc…      ┆ b8b7-bb3…   │
└──────────────┴──────────────┴──────────────┴─────────┴──────────────┴──────────────┴─────────────┘

In [143]:
 (
    data.explode("members")
    .unnest("members")
    .with_columns(
        pl.col("account_details").struct["country"].alias("country"),
        pl.col("account_details")
        .struct["registration_date"]
        .alias("registration_date"),
        pl.col("player_id").alias("player_id"),
        pl.col("stats").struct["level"].alias("level"),
        pl.col("stats").struct["experience"].alias("experience"),
        pl.col("stats").struct["total_matches"].alias("total_matches"),
        pl.col("stats").struct["win_rate"].alias("win_rate"),
        pl.col("stats").struct["playtime_hours"].alias("playtime_hours"),
        pl.col("inventory").struct["items"].alias("items"),
        today = datetime.date.today(),
    )
    .select(
        [
            pl.col("team_id").alias("team_id"),
            pl.col("created_date").str.to_date("%Y-%m-%d").alias("team_created_date"),
            pl.col("ranking").alias("team_ranking"),
            pl.col("total_winnings").alias("team_total_winnings"),
            pl.col("country").alias("player_country"),
            pl.col("registration_date").str.to_date("%Y-%m-%d").alias("player_registration_date"),
            pl.col("today").alias("today"),
            pl.col("level").alias("player_level"),
            pl.col("experience").alias("player_experience"),
            pl.col("total_matches").alias("player_total_matches"),
            pl.col("win_rate").alias("player_win_rate"),
            pl.col("playtime_hours").alias("player_playtime_hours"),
            pl.col("items")
            .list.eval(pl.element().struct["rarity"].len())
            .list.get(0, null_on_oob=True)
            .alias("numb_rare_items"),
        ]
    )
).group_by(
    ["team_id", "team_created_date", "team_ranking", "team_total_winnings"]
).agg(
    [
        pl.col("player_country").value_counts().sort(descending=True).first().struct.field('player_country').alias("most_common_country"),
        (pl.col("today") - pl.col("player_registration_date")).mean().dt.total_days().alias("avg_days_since_registration"),
        pl.col("player_level").mean().alias("avg_player_level"),
        pl.col("player_experience").mean().alias("avg_player_experience"),
        pl.col("player_total_matches").mean().alias("avg_player_total_matches"),
        pl.col("player_win_rate").mean().alias("avg_player_win_rate"),
        pl.col("player_playtime_hours").mean().alias("avg_player_playtime_hours"),
        pl.col("numb_rare_items").mean().alias("avg_numb_rare_items"),
    ]
).sort(by="team_ranking", descending=False).collect()

team_id,team_created_date,team_ranking,team_total_winnings,most_common_country,avg_days_since_registration,avg_player_level,avg_player_experience,avg_player_total_matches,avg_player_win_rate,avg_player_playtime_hours,avg_numb_rare_items
str,date,i64,i64,str,i64,f64,f64,f64,f64,f64,f64
"""89009bb7-e041-4e37-816e-45e2b2…",2024-11-24,1,732240,"""Timor-Leste""",79,51.8,79580.0,458.6,56.596,3069.0,9.0
"""7cf6999f-c35e-4069-beeb-720e6c…",2024-10-24,1,562874,"""Syrian Arab Republic""",51,34.285714,45530.285714,525.0,54.048571,2290.571429,10.285714
"""5e43eac0-5fd6-4f1b-94cd-894f90…",2024-11-23,1,603391,"""Suriname""",35,44.571429,69268.571429,581.142857,51.097143,2970.571429,10.142857
"""d9299321-699e-4c9b-a1a9-5cb37b…",2024-12-08,1,135790,"""Tunisia""",54,57.857143,93478.142857,589.714286,57.407143,2322.714286,10.285714
"""897cef00-5f71-4b2c-aa32-4cd01f…",2024-11-27,1,999945,"""Tunisia""",41,40.166667,53250.166667,366.833333,55.716667,2513.166667,8.0
…,…,…,…,…,…,…,…,…,…,…,…
"""30708f69-4428-4381-a54f-32f04c…",2024-11-18,1000,427993,"""Tokelau""",38,41.555556,62627.555556,562.0,54.453333,2387.222222,9.222222
"""677d933c-571b-4d7c-bbbd-b2bde8…",2024-10-30,1000,719648,"""Northern Mariana Islands""",37,30.333333,48175.666667,639.5,57.173333,2355.0,8.833333
"""a1138000-6663-4fd4-b86b-d5e804…",2024-12-06,1000,528504,"""Wallis and Futuna""",47,30.666667,46058.0,468.166667,55.91,2673.666667,10.833333
